<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/overlaps_and_slivers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import geopandas as gpd
from shapely import set_precision, make_valid
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. CONFIGURATION
INPUT_PATH = '/content/drive/MyDrive/your_folder/P1.shp'
OUTPUT_FIXED = '/content/drive/MyDrive/your_folder/PRUEBA1_BNG_FINAL.shp'
GRID_SIZE = 0.001  # 1mm precision
SLIVER_THRESHOLD = 0.01 # Polygons smaller than 0.01 square meters are removed

# 3. LOAD & PROJECT
gdf = gpd.read_file(INPUT_PATH)
if gdf.crs is None or gdf.crs.to_epsg() != 27700:
    gdf = gdf.to_crs("EPSG:27700")
    print("Projected to British National Grid (Meters).")

# 4. IDENTIFY INITIAL ISSUES
# Overlaps
overlaps_check = gpd.sjoin(gdf, gdf, how='inner', predicate='overlaps')
init_overlaps = len(overlaps_check[overlaps_check.index < overlaps_check['index_right']])

# Slivers (Areas smaller than the threshold)
init_slivers = len(gdf[gdf.geometry.area < SLIVER_THRESHOLD])

print(f"Initial Scan: Found {init_overlaps} overlaps and {init_slivers} slivers.")

# 5. REMOVE OVERLAPS AND SLIVERS
# Sort by area descending so large polygons "carve" the space for small ones
gdf['temp_area'] = gdf.geometry.area
gdf = gdf.sort_values(by='temp_area', ascending=False).reset_index(drop=True)

fixed_geometries = []
footprint_processed = None

print("Processing geometries... (Nuclear Option)")

for i, row in gdf.iterrows():
    # Snap to precision and ensure validity
    current_geom = make_valid(set_precision(row.geometry, grid_size=GRID_SIZE))

    if footprint_processed is None:
        clean_geom = current_geom
        footprint_processed = current_geom
    else:
        # Subtract everything already processed to ensure 0 overlap
        clean_geom = make_valid(current_geom.difference(footprint_processed))

        # Update the total footprint
        footprint_processed = make_valid(footprint_processed.union(current_geom))
        footprint_processed = set_precision(footprint_processed, grid_size=GRID_SIZE)

    fixed_geometries.append(clean_geom)

# 6. FINAL CLEANUP & EXPORT
gdf_fixed = gdf.copy()
gdf_fixed['geometry'] = fixed_geometries

# Final filters:
# 1. Remove empty geometries
# 2. Remove slivers (Area < Threshold)
# 3. Final precision snap to eliminate "floating point ghosts"
gdf_fixed = gdf_fixed[~gdf_fixed.is_empty]
gdf_fixed = gdf_fixed[gdf_fixed.geometry.area >= SLIVER_THRESHOLD]
gdf_fixed['geometry'] = gdf_fixed['geometry'].apply(lambda g: set_precision(make_valid(g), grid_size=GRID_SIZE))

# Drop temp column
if 'temp_area' in gdf_fixed.columns:
    gdf_fixed = gdf_fixed.drop(columns=['temp_area'])

# Save result
gdf_fixed.to_file(OUTPUT_FIXED)

# 7. FINAL VALIDATION REPORT
final_overlaps_check = gpd.sjoin(gdf_fixed, gdf_fixed, how='inner', predicate='overlaps')
final_overlaps = len(final_overlaps_check[final_overlaps_check.index < final_overlaps_check['index_right']])

print("-" * 30)
print(f"✅ SUCCESS: Process Complete.")
print(f"Final Count: {len(gdf_fixed)} features.")
print(f"Remaining Overlaps: {final_overlaps}")
print(f"File Saved: {OUTPUT_FIXED}")